# Qualtrics Survey Change Check

Lightweight, standalone check for whether any of the 6 Qualtrics surveys used by `qualtrics_data_extract_fetch.ipynb` / `qualtrics_fetch.ipynb` have been edited since the last time their `Qnn` field mappings were manually verified.

**What this does**: compares each survey's real `lastModified` timestamp (from the Qualtrics `/surveys` API) against a saved baseline. If a survey's `lastModified` is newer than the baseline, that means someone edited the survey (added/removed/reordered questions, etc.) since the mappings were last checked — and the hardcoded `Qnn` field tags in the other two notebooks may have drifted and need re-verifying.

**What this does NOT do**: it does not identify which specific field drifted, and it does not fix anything automatically. It is a trigger, not a diagnosis — if it flags a survey, the next step is to manually re-run the same kind of live verification used before (compare real CSV export headers / `survey-definitions` API against the `standardise_*_full()` functions).

**Independent** from `qualtrics_data_extract_fetch.ipynb` and `qualtrics_fetch.ipynb` — does not import or depend on either. Survey IDs and API config are copied here, not shared, so this notebook can be run on its own (e.g. before deciding whether to trust a run of the other two).

In [ ]:
import requests
import json
import os
from datetime import datetime

API_TOKEN = "lyf0SLF0sEvAgCx6t94UskYsmiTBfWt7ADZ3dnwn"
BASE_URL  = "https://syd1.qualtrics.com/API/v3"
headers = {"X-API-TOKEN": API_TOKEN}

SURVEYS = {
    "CSAT":           "SV_eXLP1SmIM6zjdXv",  # Customer Satisfaction Survey
    "POC":            "SV_9GjRYiNTNKsuKkR",  # POC Survey
    "HealthServices": "SV_1Yx80MMG0nDHTAF",  # Health Services
    "MentalHealth":   "SV_2hm4kwhNudIHB2t",  # Mental Health Program
    "Wellbeing":      "SV_6m5bFDluaJjxO8R",  # Health and Wellbeing Program
    "WellbeingV2":     "SV_efX6mORlYWtebXg",  # HCS NPS Questionnaire (Wellbeing V2)
}

# Baseline file lives alongside this notebook, in the same 19_Qualtrics folder —
# it records the lastModified timestamp for each survey as of the last time the
# Qnn field mappings in the other two notebooks were manually verified against
# live data.
BASELINE_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "survey_baseline.json")

# Fetch current survey metadata

In [ ]:
def fetch_current_last_modified() -> dict:
    """Returns {label: lastModified_string} for all 6 surveys, using the live /surveys API."""
    resp = requests.get(f"{BASE_URL}/surveys", headers=headers)
    resp.raise_for_status()
    elements = resp.json()["result"]["elements"]
    by_id = {s["id"]: s.get("lastModified") for s in elements}

    current = {}
    for label, survey_id in SURVEYS.items():
        if survey_id not in by_id:
            raise RuntimeError(f"[{label}] Survey ID {survey_id} not found in /surveys response — check API access/permissions.")
        current[label] = by_id[survey_id]
    return current

current_last_modified = fetch_current_last_modified()
for label, ts in current_last_modified.items():
    print(f"[{label}] lastModified = {ts}")

# Compare against baseline

If `survey_baseline.json` doesn't exist yet (first run), this just creates it from the current state — there's nothing to compare against yet, so no surveys are flagged on the very first run. Treat that first save as "I am confirming the current Qnn mappings are correct as of today."

In [ ]:
def load_baseline() -> dict:
    if not os.path.exists(BASELINE_PATH):
        return {}
    with open(BASELINE_PATH, "r", encoding="utf-8") as f:
        return json.load(f)


def save_baseline(data: dict) -> None:
    payload = {
        "_saved_at": datetime.now().isoformat(),
        "surveys": data,
    }
    with open(BASELINE_PATH, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)


baseline = load_baseline()

if not baseline:
    print("No baseline found — this looks like the first run.")
    print("Saving current lastModified values as the new baseline.")
    print("Only do this if you have just finished verifying the Qnn field mappings are correct!")
    save_baseline(current_last_modified)
    print(f"Baseline saved to {BASELINE_PATH}")
else:
    baseline_surveys = baseline.get("surveys", {})
    print(f"Baseline last saved: {baseline.get('_saved_at', 'unknown')}\n")

    changed = []
    for label, current_ts in current_last_modified.items():
        baseline_ts = baseline_surveys.get(label)
        if baseline_ts is None:
            print(f"[{label}] No baseline entry — treat as changed (never verified).")
            changed.append(label)
        elif current_ts != baseline_ts:
            print(f"[{label}] CHANGED — baseline={baseline_ts}  current={current_ts}")
            changed.append(label)
        else:
            print(f"[{label}] unchanged (lastModified={current_ts})")

    print()
    if changed:
        print(f"⚠️  WARNING: {len(changed)} survey(s) have been edited since the last verification: {', '.join(changed)}")
        print("The Qnn field mappings in qualtrics_data_extract_fetch.ipynb / qualtrics_fetch.ipynb for these surveys")
        print("may have drifted and should be re-verified against a live export before being trusted.")
    else:
        print("✅ No changes detected — all 6 surveys match the last verified baseline.")

# Update the baseline (only after you have re-verified field mappings)

Run this cell only after you have manually re-checked the Qnn mappings for any survey(s) flagged above (e.g. via a fresh `export-responses` pull + `ImportId` cross-check, the same method used to fix the CSAT/HealthServices drift issues). This resets the baseline to "current is now considered correct."

In [ ]:
# Uncomment and run only after you've actually re-verified the flagged surveys' field mappings:
# save_baseline(current_last_modified)
# print(f"Baseline updated to current state as of {datetime.now().isoformat()}")